# Instructor Solution: Math 00C: Functions, Tables, and Graphs        **Level:** 0        **Learning Objectives**        1. Understand a function as an input-output rule.2. Build tables of values.3. Connect formulas to plotted graphs.4. Recognize slope as repeated change.

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "math_00_functions_and_graphs",    "level": 0,    "title": "Math 00C: Functions, Tables, and Graphs",    "prerequisites": [        "py.variables.functions",        "math.pre_algebra.order_operations"    ],    "skills_taught": [        "math.functions.input_output",        "math.functions.graph_tables"    ],    "skills_practiced": [        "py.variables.functions",        "math.pre_algebra.order_operations"    ],    "next_notebook": "math_01_algebra_linear_equations.ipynb"}        SKILLS = [        {"id": "math.functions.input_output", "title": "math.functions.input_output", "notebook": "math_00_functions_and_graphs.ipynb", "level": 0},{"id": "math.functions.graph_tables", "title": "math.functions.graph_tables", "notebook": "math_00_functions_and_graphs.ipynb", "level": 0},{"id": "py.variables.functions", "title": "py.variables.functions", "notebook": "math_00_functions_and_graphs.ipynb", "level": 0},{"id": "math.pre_algebra.order_operations", "title": "math.pre_algebra.order_operations", "notebook": "math_00_functions_and_graphs.ipynb", "level": 0}        ]

## Before You Learn: Pull From MemoryAnswer these before reading the lesson. The point is retrieval, not perfection.

**Recall function_memory.** If f(x)=2x+1, what is f(3)?

In [ ]:
answer = ""  # type your answer inside the quotesshort_answer_check(    'function_memory',    answer,    ['7'],    skill_id='math.functions.input_output',    confidence=3,    store=store,)

**Recall slope_memory.** In y=3x+2, what number tells you the repeated change in y?

In [ ]:
answer = ""  # type your answer inside the quotesshort_answer_check(    'slope_memory',    answer,    ['3', 'slope'],    skill_id='math.functions.graph_tables',    confidence=3,    store=store,)

## Micro-Lesson            A function is a rule that turns an input into an output. A table shows several inputs and outputs.            A graph draws those pairs as points. When a function is linear, equal steps in `x` create equal            changes in `y`.

## Worked Example            For `f(x) = 2x + 1`, the input `x = 3` gives `2 * 3 + 1 = 7`.            The point `(3, 7)` belongs on the graph.

## Faded Example            For `g(x) = 4x - 2`, the input `x = 5` gives `4 * ___ - 2 = ___`.

## Interactive Visualization

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport ipywidgets as widgetsfrom IPython.display import displayslope = widgets.IntSlider(value=2, min=-5, max=5, description='slope')intercept = widgets.IntSlider(value=1, min=-10, max=10, description='intercept')out = widgets.Output()def plot_line(change=None):    with out:        out.clear_output()        xs = np.arange(-5, 6)        ys = slope.value * xs + intercept.value        plt.figure(figsize=(5, 3))        plt.axhline(0, color='black', linewidth=0.5)        plt.axvline(0, color='black', linewidth=0.5)        plt.plot(xs, ys, marker='o')        plt.title(f'y = {slope.value}x + {intercept.value}')        plt.grid(True)        plt.show()slope.observe(plot_line, names='value')intercept.observe(plot_line, names='value')display(slope, intercept, out)plot_line()

## Independent Practice

### Write a Linear FunctionWrite `f(x)` for the rule `2x + 1`.

In [ ]:
def f(x):    return 2 * x + 1

In [ ]:
code_check(    'f',    f,    [((0,), 1), ((3,), 7), ((-2,), -3)],    skill_id='math.functions.input_output',    confidence=3,    store=store,)

In [ ]:
hint_box(    'Hints for Write a Linear Function',    ['Use the input variable `x`.', 'Double the input, then add one.', '`return 2 * x + 1`.'],    unlock=False,)

**Instructor note:** The full solution is included above. The student version receives scaffolded code, active checks, and staged hints.

### Build a TableWrite `build_table(fn, xs)` to return pairs `(x, fn(x))`.

In [ ]:
def build_table(fn, xs):    table = []    for x in xs:        table.append((x, fn(x)))    return table

In [ ]:
code_check(    'build_table',    build_table,    [((lambda x: 2*x + 1, [0, 1, 2]), [(0, 1), (1, 3), (2, 5)])],    skill_id='math.functions.graph_tables',    confidence=3,    store=store,)

In [ ]:
hint_box(    'Hints for Build a Table',    ['A table is a list of input-output pairs.', 'Loop through `xs` and call the function on each input.', '`table.append((x, fn(x)))` inside the loop.'],    unlock=False,)

**Instructor note:** The full solution is included above. The student version receives scaffolded code, active checks, and staged hints.

## Transfer ChallengeCreate a table and graph for a taxi fare rule: base fee plus price per kilometer.

## End-of-Notebook ReviewBefore leaving, retrieve the most important ideas once more.

In [ ]:
mastery_dashboard(store=store, skills=SKILLS, next_notebook='math_01_algebra_linear_equations.ipynb')